# LLM-to-Database Strategy

This notebook covers the basics of connecting LLM workflows to databases:

- Text-to-SQL with a local SQLite file
- When to use SQL, vector search, or hybrid retrieval
- Structured output into database rows
- LLM as a simple query planner / intent router

LangChain’s SQL agent tutorial shows the SQL workflow for querying databases and notes the risk of model-generated SQL, so permissions should be narrowly scoped. Retrieval docs explain that LLMs have finite context and static knowledge, and retrieval helps by fetching relevant external knowledge at query time. Vector store docs describe vector stores as embedded data structures used for similarity search. Structured output docs show how LangChain can return predictable JSON or Pydantic-shaped data.


## Learning goals

By the end of this notebook, you should be able to:

1. Understand the basic idea behind text-to-SQL.
2. Decide when SQL is the right store, when vector search is the right store, and when both are useful.
3. Parse model output into a clean database row.
4. Build a small intent router that sends a prompt to the right data source.


## 1) Install dependencies

This notebook keeps the tooling simple.

```bash
pip install -U sqlmodel aiosqlite langchain langchain-groq pydantic python-dotenv
```

You can use either raw `sqlite3` or SQLModel. For the very first version, raw SQLite is enough.


In [1]:
%pip install -qU sqlmodel aiosqlite langchain langchain-groq pydantic python-dotenv


Note: you may need to restart the kernel to use updated packages.


  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-aws 1.1.0 requires numpy<3,>=2.3.2; python_version >= "3.12", but you have numpy 1.26.4 which is incompatible.
langchain-community 0.4.1 requires numpy>=2.1.0; python_version >= "3.13", but you have numpy 1.26.4 which is incompatible.
langchain-tests 1.1.4 requires numpy>=2.1.0; python_version >= "3.13", but you have numpy 1.26.4 which is incompatible.
langchain-tests 1.1.4 requires pytest<9.0.0,>=7.0.0, but you have pytest 9.0.3 which is incompatible.
sagemaker 2.245.0 requires packaging<25,>=23.0, but you have packaging 26.0 which is incompatible.
sagemaker-mlops 1.5.0 requires sagemaker-core>=2.5.0, but you have sagemaker-core 1.0.77 which is incompatible.
sagemaker-serve 1.5.0 requires sagemaker-core>=2.5.0, but you have sagemaker-core 1.0.77 which is incompatible

## 2) The three basic data paths

### Use SQL when:
- the data is structured
- the fields are known in advance
- you need filters, counts, joins, or exact lookups

### Use vector search when:
- the information is mostly text
- the meaning matters more than exact fields
- you want semantic similarity search

### Use hybrid retrieval when:
- part of the answer is structured
- part of the answer is semantic text
- you want SQL filters plus vector search together

LangChain’s retrieval docs say retrieval helps when LLMs cannot hold all content in context, and vector store docs describe similarity search over embedded data.


## 3) A tiny SQLite database

We will create a very small local database for:
- prompt history
- token usage
- model latency

This is enough for a basic LLM-to-database workflow.


In [2]:
import sqlite3
from pathlib import Path
from datetime import datetime

db_path = Path("llm_strategy_basic.db")

conn = sqlite3.connect(db_path)
cur = conn.cursor()

cur.execute('''
CREATE TABLE IF NOT EXISTS prompt_history (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    user_prompt TEXT NOT NULL,
    llm_response TEXT,
    created_at TEXT NOT NULL
)
''')

cur.execute('''
CREATE TABLE IF NOT EXISTS token_usage (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    request_name TEXT NOT NULL,
    input_tokens INTEGER DEFAULT 0,
    output_tokens INTEGER DEFAULT 0,
    total_tokens INTEGER DEFAULT 0,
    created_at TEXT NOT NULL
)
''')

cur.execute('''
CREATE TABLE IF NOT EXISTS latency_log (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    request_name TEXT NOT NULL,
    model_name TEXT NOT NULL,
    latency_ms REAL NOT NULL,
    created_at TEXT NOT NULL
)
''')

conn.commit()
conn.close()

print("Created database:", db_path.resolve())


Created database: D:\personal_docs\course-ai\module-1\db\llm_strategy_basic.db


## 4) Raw SQL text-to-SQL idea

A text-to-SQL flow usually looks like this:

1. User asks a question.
2. The LLM identifies the target table and fields.
3. The LLM generates a SQL query.
4. The app runs the query on SQLite.
5. The app returns the result.

LangChain’s SQL agent tutorial follows a similar pattern: inspect schemas, choose relevant tables, generate a query, double-check it, execute it, and answer from the results. It also warns that model-generated SQL should run with narrow permissions.


In [3]:
# Simple, safe SQL example without any LLM call
conn = sqlite3.connect(db_path)
cur = conn.cursor()

cur.execute(
    "INSERT INTO prompt_history (user_prompt, llm_response, created_at) VALUES (?, ?, ?)",
    ("Show me recent prompts", "Here are the recent prompts...", datetime.utcnow().isoformat())
)
conn.commit()

cur.execute("SELECT id, user_prompt, llm_response FROM prompt_history ORDER BY id DESC LIMIT 5")
rows = cur.fetchall()
print(rows)

conn.close()


[(1, 'Show me recent prompts', 'Here are the recent prompts...')]


C:\Users\MadhiarasanM\AppData\Local\Temp\ipykernel_29584\1583324255.py:7: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  ("Show me recent prompts", "Here are the recent prompts...", datetime.utcnow().isoformat())


## 5) Query planner basics

At a high level, an LLM can act as a query planner.

It does not need to be a full agent in this notebook. A simple planner only needs to decide:

- SQL
- vector search
- hybrid
- direct answer

That is enough for a first pass before you move into deeper RAG and agent workflows.


In [4]:
def choose_store(user_prompt: str) -> str:
    prompt = user_prompt.lower()

    sql_keywords = ["count", "total", "sum", "average", "latest", "how many", "by date", "token", "latency"]
    vector_keywords = ["meaning", "similar", "find documents", "semantic", "text search", "paragraph", "notes"]
    hybrid_keywords = ["and", "together", "both", "plus"]

    if any(k in prompt for k in hybrid_keywords) and any(k in prompt for k in sql_keywords + vector_keywords):
        return "hybrid"
    if any(k in prompt for k in sql_keywords):
        return "sql"
    if any(k in prompt for k in vector_keywords):
        return "vector"
    return "direct"

examples = [
    "How many prompts were logged yesterday?",
    "Find similar meeting notes about invoices.",
    "Show metadata and similar notes together.",
    "Explain what prompt history means.",
]

for q in examples:
    print(q, "->", choose_store(q))


How many prompts were logged yesterday? -> sql
Find similar meeting notes about invoices. -> vector
Show metadata and similar notes together. -> hybrid
Explain what prompt history means. -> direct


## 6) SQL vs vector vs hybrid

### SQL
Best for:
- prompt logs
- token counts
- model latency
- structured metadata

### Vector search
Best for:
- notes
- documents
- policy text
- semantic question answering

### Hybrid
Best for:
- structured filters plus semantic matching
- search over documents with metadata filters
- cases where both exact fields and meaning matter

LangChain’s retrieval docs explain that existing systems can be used as a tool or queried and passed back as context, and the vector store docs show similarity search plus metadata filtering. citeturn885159view0turn885159view1


## 7) Structured output into database rows

A simple pattern is:

1. Ask the model for structured output.
2. Validate it with a Pydantic schema.
3. Insert the validated object into SQLite.

LangChain’s structured output docs say the output can be returned as JSON objects, Pydantic models, or dataclasses, and the data is validated before use. citeturn885159view2


In [5]:
from pydantic import BaseModel, Field, ValidationError

class PromptRecord(BaseModel):
    user_prompt: str = Field(...)
    category: str = Field(...)
    answer: str = Field(...)

# Example of model output already shaped like structured data
raw_output = {
    "user_prompt": "How many prompts were logged yesterday?",
    "category": "sql",
    "answer": "Use a SQL query on the prompt_history table.",
}

try:
    record = PromptRecord(**raw_output)
    print(record)
except ValidationError as e:
    print(e)


user_prompt='How many prompts were logged yesterday?' category='sql' answer='Use a SQL query on the prompt_history table.'


In [6]:
# Insert the validated row into SQLite
conn = sqlite3.connect(db_path)
cur = conn.cursor()

record = PromptRecord(
    user_prompt="How many prompts were logged yesterday?",
    category="sql",
    answer="Use a SQL query on the prompt_history table.",
)

cur.execute(
    "INSERT INTO prompt_history (user_prompt, llm_response, created_at) VALUES (?, ?, ?)",
    (record.user_prompt, record.answer, datetime.utcnow().isoformat())
)

conn.commit()
conn.close()

print("Inserted structured record into prompt_history")


Inserted structured record into prompt_history


C:\Users\MadhiarasanM\AppData\Local\Temp\ipykernel_29584\1782940858.py:13: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  (record.user_prompt, record.answer, datetime.utcnow().isoformat())


## 8) Optional: LLM-backed structured output

If you later connect a model, keep the same schema and only swap the input source.

A simple pattern is:

- prompt the model for JSON
- parse into `PromptRecord`
- store in SQLite

That gives you a stable database row format even if the prompt text changes.


In [8]:
# Optional skeleton for a Groq-backed call later.
# Keep this cell as a template for when you want to connect your API key.
from dotenv import load_dotenv
load_dotenv()
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
prompt = ChatPromptTemplate.from_messages([
    ("system", "Return only JSON with keys: user_prompt, category, answer."),
    ("user", "{question}")
])
chain = prompt | llm | StrOutputParser()
print(chain.invoke({"question": "How many prompts were logged yesterday?"}))
print("LLM template cell ready.")


```json
{
  "user_prompt": "How many prompts were logged yesterday?",
  "category": "unknown",
  "answer": "I'm not able to access that information."
}
```
LLM template cell ready.


## 9) Hybrid routing logic

A planner can route to a store based on the user intent.

This version stays intentionally simple:

- SQL for structured metrics
- vector for semantic text
- hybrid for mixed cases
- direct answer for general explanations


In [9]:
def route_request(user_prompt: str) -> str:
    decision = choose_store(user_prompt)

    if decision == "sql":
        return "Route to SQLite"
    if decision == "vector":
        return "Route to vector search"
    if decision == "hybrid":
        return "Route to SQL + vector search"
    return "Answer directly"

for q in examples:
    print(q, "->", route_request(q))


How many prompts were logged yesterday? -> Route to SQLite
Find similar meeting notes about invoices. -> Route to vector search
Show metadata and similar notes together. -> Route to SQL + vector search
Explain what prompt history means. -> Answer directly


## 10) Why this matters

This is the small version of the larger architecture:

- the LLM decides where the data lives
- the app executes the right retrieval step
- the result is returned in a consistent structure

That is the core idea behind an LLM query planner.


## 11) Keep it safe and simple

For early implementations:

- use read-only SQL where possible
- keep the schema narrow
- validate structured outputs before insert
- use vector search only for text that needs semantic matching
- do not overbuild agents before the basic routing is working

LangChain’s SQL tutorial explicitly notes that model-generated SQL has risks and permissions should be scoped narrowly. citeturn885159view3


## 12) What comes next

This notebook is intentionally basic.

Later, you can expand it into:

- real text-to-SQL with LangChain SQL tools
- a proper vector store for document search
- hybrid retrieval pipelines
- agent-based query planning
- evaluation and tracing

For now, the goal is just to understand the decision boundary between the data stores.


## Key takeaways

- SQL is best for structured data and metrics.
- Vector search is best for semantic text matching.
- Hybrid retrieval is useful when both matter.
- Structured output helps you write clean rows into SQLite.
- A simple query planner can route requests before you build deeper RAG or agent workflows. 


## References

- SQL agent: https://docs.langchain.com/oss/python/langchain/sql-agent
- Vector store integrations: https://docs.langchain.com/oss/python/integrations/vectorstores/index
- Retrieval: https://docs.langchain.com/oss/python/langchain/retrieval#building-a-knowledge-base
- Structured output: https://docs.langchain.com/oss/python/langchain/structured-output
